In [1]:
#=================================================================================#
#==============DATA INGESTION, DATA CLEANING AND DATA PROCESSING==================#
#=================================================================================#

In [2]:
#IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
#LOAD DATASET USING PANDAS

df = pd.read_csv(r"C:\Users\nivet\Downloads\Warehouse_Dataset.csv")
print("Dataset Loaded Successfully")

print("Rows: ",df.shape[0])
print("Columns: ",df.shape[1])

Dataset Loaded Successfully
Rows:  1000
Columns:  10


In [4]:
original_df= df.copy()

print("Original dataset shape:",original_df.shape)
print("Original missing values:", original_df.isnull().sum().sum())
print("Original duplicate rows:", original_df.duplicated().sum())

Original dataset shape: (1000, 10)
Original missing values: 565
Original duplicate rows: 0


In [5]:
df.head()

,Product ID,Product Name,Category,Warehouse,Location,Quantity,Price,Supplier,Status,Last Restocked
0,1102,gadget y,ELECTRONICS,Warehouse 2,Aisle 1,300,9.99,Supplier C,In Stock,NaN
1,1435,gadget y,ELECTRONICS,Warehouse 2,Aisle 4,two hundred,19.99,Supplier C,Out of Stock,NaN
2,1860,widget a,CLOTHING,Warehouse 2,Aisle 3,100,19.99,Supplier B,In Stock,20-12-2022
3,1270,gadget z,TOYS,Warehouse 2,Aisle 4,50,49.99,Supplier B,In Stock,20-12-2022
4,1106,widget a,FURNITURE,Warehouse 3,Aisle 3,two hundred,9.99,Supplier D,Out of Stock,25-04-2023


In [6]:
df.info()
df.describe(include ="all")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Product ID      1000 non-null   int64  
 1   Product Name    1000 non-null   object 
 2   Category        1000 non-null   object 
 3   Warehouse       1000 non-null   object 
 4   Location        1000 non-null   object 
 5   Quantity        842 non-null    object 
 6   Price           793 non-null    float64
 7   Supplier        1000 non-null   object 
 8   Status          1000 non-null   object 
 9   Last Restocked  800 non-null    object 
dtypes: float64(1), int64(1), object(8)
memory usage: 78.3+ KB


,Product ID,Product Name,Category,Warehouse,Location,Quantity,Price,Supplier,Status,Last Restocked
count,1000.000000,1000,1000,1000,1000,842,793.000000,1000,1000,800
unique,NaN,6,4,3,5,5,NaN,4,3,4
top,NaN,gadget y,FURNITURE,Warehouse 1,Aisle 3,300,NaN,Supplier B,In Stock,20-12-2022
freq,NaN,177,265,349,211,177,NaN,288,340,218
mean,1503.929000,NaN,NaN,NaN,NaN,NaN,28.085839,NaN,NaN,NaN
std,289.998108,NaN,NaN,NaN,NaN,NaN,14.686312,NaN,NaN,NaN
min,1000.000000,NaN,NaN,NaN,NaN,NaN,9.990000,NaN,NaN,NaN
25%,1242.750000,NaN,NaN,NaN,NaN,NaN,19.990000,NaN,NaN,NaN
50%,1505.000000,NaN,NaN,NaN,NaN,NaN,29.990000,NaN,NaN,NaN
75%,1757.250000,NaN,NaN,NaN,NaN,NaN,49.990000,NaN,NaN,NaN


In [7]:
print("Shape:", df.shape)
print("\nColumn names:",df.columns.tolist())

Shape: (1000, 10)

Column names: ['Product ID', 'Product Name', 'Category', 'Warehouse', 'Location', 'Quantity', 'Price', 'Supplier', 'Status', 'Last Restocked']


In [8]:
#MISSING VALUES
missing = df.isnull().sum()
print("The Missing Values in Each Column")
print(missing)

The Missing Values in Each Column
Product ID          0
Product Name        0
Category            0
Warehouse           0
Location            0
Quantity          158
Price             207
Supplier            0
Status              0
Last Restocked    200
dtype: int64


In [9]:
#MISSING PERCENTAGE
missing_percent=(df.isnull().sum()/ len(df))*100

missing_summary=pd.DataFrame({
    "Missing Count" : df.isnull().sum(),
    "Missing Percentage" : missing_percent
})
missing_summary

,Missing Count,Missing Percentage
Product ID,0,0.0
Product Name,0,0.0
Category,0,0.0
Warehouse,0,0.0
Location,0,0.0
Quantity,158,15.8
Price,207,20.7
Supplier,0,0.0
Status,0,0.0
Last Restocked,200,20.0


In [10]:
#FIX QUANTITY

quantity_map={
    "two hundred" : 200
}
df["Quantity"]=df["Quantity"].replace(quantity_map)
df["Quantity"]=pd.to_numeric(df["Quantity"],errors="coerce")

In [11]:
df["Quantity"].dtype

dtype('float64')

In [12]:
#HANDLE MISSING QUANTITIES
quantity_median= df["Quantity"].median()
df["Quantity"]=df["Quantity"].fillna(quantity_median)

#Check for null values in Quantity Column
print(df["Quantity"].isnull().sum())

0


In [13]:
#HANDLE MISSING PRICE
price_median= df["Price"].median()
df["Price"]=df["Price"].fillna(price_median)

#Check for null values in Price Column
print(df["Price"].isnull().sum())

0


In [14]:
df["Last Restocked"] = pd.to_datetime(
    df["Last Restocked"],
    format="%d-%m-%Y",
    errors="coerce"
)

df["Last Restocked"] = df["Last Restocked"].dt.strftime("%d-%m-%Y")

df["Last Restocked"] = df["Last Restocked"].fillna("Not Available")

In [15]:
df.isnull().sum()

Product ID        0
Product Name      0
Category          0
Warehouse         0
Location          0
Quantity          0
Price             0
Supplier          0
Status            0
Last Restocked    0
dtype: int64

In [16]:
#STANDARDIZE PRODUCT NAME
df["Product Name"]=df["Product Name"].str.strip().str.title()
print(df["Product Name"])

0      Gadget Y
1      Gadget Y
2      Widget A
3      Gadget Z
4      Widget A
         ...   
995    Widget B
996    Gadget Y
997    Gadget Z
998    Widget C
999    Widget A
Name: Product Name, Length: 1000, dtype: object


In [17]:
text_columns=["Category","Warehouse","Location","Supplier","Status"]

for column in text_columns:
    df[column]=df[column].astype(str).str.strip()

In [18]:
df["Category"]=df["Category"].str.title()
print(df["Category"])

0      Electronics
1      Electronics
2         Clothing
3             Toys
4        Furniture
          ...     
995      Furniture
996    Electronics
997           Toys
998      Furniture
999       Clothing
Name: Category, Length: 1000, dtype: object


In [19]:
#DUPLICATE HANDLING

duplicate_count= df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [20]:
df=df.drop_duplicates()

In [21]:
df.duplicated().sum()

0

In [22]:
#========================================================================#
#=======================FEATURE ENGINEERING==============================#
#========================================================================#

In [23]:
#FEATURE 1 : INVENTORY VALUE

df["Inventory Value"]= df["Quantity"] * df["Price"]
print("Inventory Value:")
print(df["Inventory Value"])

Inventory Value:
0      2997.0
1      3998.0
2      1999.0
3      2499.5
4      1998.0
        ...  
995    2999.0
996    5997.0
997    1498.5
998    4999.0
999    4999.0
Name: Inventory Value, Length: 1000, dtype: float64


In [27]:
#FEATURE 2 : STOCK STATUS CATEGORY

status_mapping={
    "In Stock" : 1,
    "Low Stock" : 0.5,
    "Out of Stock" : 0
}

df["Stock Score"] = df["Status"].map(status_mapping)
print("Stock Score : ")
print(df["Stock Score"])

Stock Score : 
0      1.0
1      0.0
2      1.0
3      1.0
4      0.0
      ... 
995    1.0
996    1.0
997    0.5
998    0.5
999    0.0
Name: Stock Score, Length: 1000, dtype: float64


In [25]:
#OUTLIER DETECTION

Q1=df["Quantity"].quantile(0.25)
Q3=df["Quantity"].quantile(0.75)

IQR=Q3-Q1

lower_bound=Q1 - 1.5 *IQR
upper_bound=Q3 + 1.5 *IQR

outliers=df[
    (df["Quantity"] < lower_bound) | (df["Quantity"] > upper_bound)
    ]

print("Quanitity Outliers: ", len(outliers))

Quanitity Outliers:  0


In [28]:
#COMPARISON

comparison = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Missing Values",
        "Duplicate Rows"
    ],
    "Before Cleaning": [
        original_df.shape[0],
        original_df.shape[1],
        original_df.isnull().sum().sum(),
        original_df.duplicated().sum()
    ],
    "After Cleaning": [
        df.shape[0],
        df.shape[1],
        df.isnull().sum().sum(),
        df.duplicated().sum()
    ]
})

comparison

,Metric,Before Cleaning,After Cleaning
0,Rows,1000,1000
1,Columns,10,12
2,Missing Values,565,0
3,Duplicate Rows,0,0


In [29]:
#EXPORT THE FINAL DATASET

df.to_csv("Clean_Warehouse_dataset.csv", index=False)
print("Cleaned Dataset Exported Successfully")

Cleaned Dataset Exported Successfully
